In [1]:
!pip uninstall numpy -y
!pip install "numpy<2"

!pip install scikit-surprise

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 65.8 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 2.3 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554988 sha256=f2c3074f0bd813075d312e04f675210833c2c1de5fb38dc96b4ec421cd3d510e
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [1]:
import pandas as pd

df = pd.read_csv("anime_master_ready.csv")
df['score'] = pd.to_numeric(df['score'], errors='coerce')
df['score'] = df['score'].fillna(df['score'].mean())
print(df['score'].dtype)
print(df.shape)
df.head()

float64
(24905, 17)


,anime_id,title,genres,synopsis,score,members,favorites,type,tags,watching,completed,on_hold,dropped,plan_to_watch,studio,year,content
0,1,cowboy bebop,"Action, Award Winning, Sci-Fi","Crime is timeless. By the year 2071, humanity ...",8.75,1771505,78525,TV,"Action, Adventure, Drama, Sci Fi, Bounty Hunte...",105808.0,718161.0,71513.0,26678.0,329800.0,Sunrise,"Apr 3, 1998","action, award winning, sci-fi action, adventur..."
1,5,cowboy bebop: tengoku no tobira,"Action, Sci-Fi","Another day, another bounty—such is the life o...",8.38,360978,1448,Movie,NaN,4143.0,208333.0,1935.0,770.0,57964.0,NaN,NaN,"action, sci-fi another day, another bounty—su..."
2,6,trigun,"Action, Adventure, Sci-Fi","Vash the Stampede is the man with a $$60,000,0...",8.22,727252,15035,TV,"Action, Adventure, Comedy, Drama, Shounen, Des...",29113.0,343492.0,25465.0,13925.0,146918.0,NaN,NaN,"action, adventure, sci-fi action, adventure, c..."
3,7,witch hunter robin,"Action, Drama, Mystery, Supernatural",Robin Sena is a powerful craft user drafted in...,7.25,111931,613,TV,"Fantasy, Noir, Psychic Powers, Supernatural, S...",4300.0,46165.0,5121.0,5378.0,33719.0,Sunrise,"Jul 3, 2002","action, drama, mystery, supernatural fantasy, ..."
4,8,bouken ou beet,"Adventure, Fantasy, Supernatural",It is the dark century and the people are suff...,6.94,15001,14,TV,NaN,642.0,7314.0,766.0,1108.0,3394.0,Toei Animation,"Sep 30, 2004","adventure, fantasy, supernatural it is the da..."


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=20000
)

tfidf_matrix = tfidf.fit_transform(df["content"])

print(tfidf_matrix.shape)

(24905, 20000)


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(tfidf_matrix)

In [4]:
indices = pd.Series(df.index, index=df["title"]).drop_duplicates()

In [5]:
def clean_results(results):

    unwanted = [
        "pv", "preview", "special", "recap",
        "final", "rewrite", "summary"
    ]

    mask = ~results['title'].str.contains(
        "|".join(unwanted), case=False, na=False
    )

    return results[mask]

In [6]:
def recommend_anime(title, top_n=10):

    title = title.lower()

    if title not in indices:
        return "Anime not found"

    idx = indices[title]

    sim_scores = list(enumerate(similarity[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n*4]

    anime_indices = [i[0] for i in sim_scores]

    results = df.iloc[anime_indices][['title','score']]

    results = clean_results(results)

    results = results.sort_values(by="score", ascending=False)

    return results.head(top_n)

In [7]:
recommend_anime("death note")

,title,score
12160,detective conan: episode one - the great detec...,8.24
9710,death parade,8.15
15835,id:invaded,7.84
3146,soul eater,7.84
12638,kakegurui,7.24
8332,"ore no nounai sentakushi ga, gakuen love comed...",7.16
518,yami no matsuei,7.03
14178,strike the blood iii,7.01
11770,strike the blood ii,6.98
16082,strike the blood iv,6.97


In [8]:
print(df.columns)

Index(['anime_id', 'title', 'genres', 'synopsis', 'score', 'members',
       'favorites', 'type', 'tags', 'watching', 'completed', 'on_hold',
       'dropped', 'plan_to_watch', 'studio', 'year', 'content'],
      dtype='object')


In [9]:
df['score'] = pd.to_numeric(df['score'], errors='coerce')
df['members'] = pd.to_numeric(df['members'], errors='coerce')
df['favorites'] = pd.to_numeric(df['favorites'], errors='coerce')

In [10]:
df['score'] = df['score'].fillna(df['score'].mean())

In [11]:
def recommend_by_preferences(
        genre=None,
        anime_type=None,
        studio=None,
        min_score=0,
        min_members=0,
        top_n=10):

    results = df.copy()

    if genre:
        results = results[
            results['genres'].str.contains(
                genre,
                case=False,
                na=False
            )
        ]

    if anime_type:
        results = results[
            results['type'].str.lower() == anime_type.lower()
        ]

    if studio:
        results = results[
            results['studio'].str.contains(
                studio,
                case=False,
                na=False
            )
        ]

    if min_score:
        results = results[
            results['score'] >= min_score
        ]

    if min_members:
        results = results[
            results['members'] >= min_members
        ]

    results = results.sort_values(
        by=['score','members'],
        ascending=False
    )

    return results[['title','score','members']].head(top_n)

In [12]:
recommend_by_preferences(
    genre="Action",
    anime_type="TV",
    min_score=8
)

,title,score,members
3961,fullmetal alchemist: brotherhood,9.10,3176556
16617,bleach: sennen kessen-hen,9.07,445198
9880,gintama°,9.06,595767
14865,shingeki no kyojin season 3 part 2,9.05,2104016
6456,hunter x hunter (2011),9.04,2656870
5989,gintama',9.04,525688
7240,gintama': enchousen,9.03,309261
12179,gintama.,8.98,297857
833,gintama,8.94,1023068
2647,code geass: hangyaku no lelouch r2,8.91,1699634


In [13]:
recommend_by_preferences(
    genre="Romance",
    anime_type="Movie",
    min_score=8
)

,title,score,members
22704,kaguya-sama wa kokurasetai: first kiss wa owar...,8.87,180281
404,howl no ugoku shiro,8.66,1253703
14750,seishun buta yarou wa yumemiru shoujo no yume ...,8.60,665114
13343,kimi no suizou wo tabetai,8.56,895776
3568,kara no kyoukai movie 5: mujun rasen,8.53,230852
21269,fruits basket: prelude,8.45,91498
13846,saenai heroine no sodatekata fine,8.43,172613
16259,josee to tora to sakana-tachi,8.41,381221
4008,kara no kyoukai movie 7: satsujin kousatsu (go),8.39,200492
15056,tenki no ko,8.29,928427


In [14]:
recommend_by_preferences(
    genre="Sci-Fi",
    min_members=500000
)

,title,score,members
5667,steins;gate,9.07,2440369
9880,gintama°,9.06,595767
5989,gintama',9.04,525688
833,gintama,8.94,1023068
2647,code geass: hangyaku no lelouch r2,8.91,1699634
0,cowboy bebop,8.75,1771505
1431,code geass: hangyaku no lelouch,8.70,2151722
12435,made in abyss,8.66,1294124
1822,tengen toppa gurren lagann,8.63,1547735
4956,suzumiya haruhi no shoushitsu,8.61,590160


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
ratings = pd.read_csv(
    "/content/drive/MyDrive/rating_complete.csv",
    usecols=["user_id","anime_id","rating"],
)

print(ratings.shape)

(57633278, 3)


In [18]:
ratings = ratings.sample(10000000, random_state=42)

print(ratings.shape)

(10000000, 3)


In [19]:
from surprise import Dataset, Reader

reader = Reader(rating_scale=(1,10))

data = Dataset.load_from_df(
    ratings[['user_id','anime_id','rating']],
    reader
)

In [20]:
from surprise import SVD
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(data, test_size=0.2)

model = SVD()

model.fit(trainset)

In [21]:
def hybrid_recommend(title, user_id, top_n=10):

    idx = indices[title]

    sim_scores = list(enumerate(similarity[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:50]

    anime_indices = [i[0] for i in sim_scores]

    candidates = df.iloc[anime_indices]

    scores = []

    for _, row in candidates.iterrows():

        collab_score = model.predict(user_id, row["anime_id"]).est

        popularity = (row["members"] + row["favorites"]) / 1e6

        final = (
        0.6 * collab_score +
        0.3 * row["score"] +
        0.1 * popularity * 10
    )

        scores.append((row["title"], round(final,2)))
        scores = [s for s in scores if s[0] != title]
    scores.sort(key=lambda x: x[1], reverse=True)

    return scores[:top_n]

In [22]:
hybrid_recommend("death note", 255419)

[('death parade', 9.38),
 ('soul eater', 8.65),
 ('kakegurui', 8.18),
 ('id:invaded', 7.75),
 ('detective conan: episode one - the great detective turned small', 7.41),
 ('koyomimonogatari', 7.33),
 ('strike the blood iv', 6.59),
 ('strike the blood final', 6.52),
 ('wonder light', 6.43),
 ('bakemonogatari recap', 6.39)]

model.predict(uid=255419, iid=4059)

anime_ids = df["anime_id"].unique()

def recommend_for_user(user_id, anime_ids, model, top_n=10):

    predictions = [
        (anime_id, model.predict(user_id, anime_id).est)
        for anime_id in anime_ids
    ]

    predictions.sort(key=lambda x: x[1], reverse=True)

    return predictions[:top_n]

def recommend_titles(user_id):

    recs = recommend_for_user(user_id, anime_ids, model)

    results = []

    for anime_id, score in recs:
        title = df[df["anime_id"] == anime_id]["title"].values[0]
        results.append((title, score))

    return results

recommend_titles(255419)

In [23]:
import pickle

# Save SVD collaborative model
with open("svd_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save TF-IDF vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

# Save cosine similarity matrix
# with open("cosine_sim.pkl", "wb") as f:
#     pickle.dump(similarity, f)

print("Models saved successfully")

Models saved successfully


In [25]:
import gzip

In [29]:
with gzip.open("svd_model.pkl.gz","wb") as f:
    pickle.dump(model,f)

In [30]:
import sys
print(sys.getsizeof(similarity))

4962072328
